>  이 파일은 해설 버전입니다.

# 4. 검색 심화 - 시맨틱 검색 & 하이브리드 검색

## 학습 목표
- Dense Vector와 k-NN 검색의 원리를 이해한다
- 키워드+벡터 결합: 하이브리드 검색(RRF)을 학습한다
- Python 클라이언트를 활용한 검색 파이프라인을 구현한다

## 4.1 OpenSearch 접속 설정
### OpenSearch 실습 환경 접속 가이드

1. 필요 패키지 설치

```bash
pip install opensearch-py openai
```

2. OpenSearch 서버 접속 설정

```python
from opensearchpy import OpenSearch

# ★ 본인 번호로 변경 (0~30) ★
STUDENT_NUMBER = 0

# 서버 접속 정보
OPENSEARCH_HOST = "sdsos.baeum.ai.kr"
OPENSEARCH_PORT = 443
OPENSEARCH_USER = "sdsrag"
OPENSEARCH_PASS = "Student.Rag1!"

# 클라이언트 생성
client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=True,
    verify_certs=True,
    ssl_show_warn=False,
)

# 접속 확인
print(client.info()["version"]["number"])
```

3. 인덱스 이름 규칙

```python
# 본인 번호에 맞는 인덱스 사용 (다른 번호 인덱스 사용 금지)
INDEX_NAME = f"student_{STUDENT_NUMBER:02d}_company_docs"
# 예: student_01_company_docs, student_02_company_docs, ...
```

4. 접속 확인 테스트

```python
# 클러스터 상태 확인
print(client.cluster.health()["status"])

# 내 인덱스 목록 확인
print(client.cat.indices(index="student_*", format="json"))
```

5. OpenSearch Dashboards (웹 UI)

- 주소: https://sdsos.baeum.ai.kr/dashboard
- 계정: sdsrag / Student.Rag1!
- 로그인 후 좌측 메뉴 → Discover → student_* 인덱스 패턴 선택
- 노트북에서 인덱싱한 문서를 검색/시각화할 수 있음

6. 주의사항

- STUDENT_NUMBER를 반드시 본인 번호로 변경할 것
- 다른 학생의 인덱스(student_XX_*)에는 접근할 수 없음
- OpenAI API 키는 각자 본인 키를 사용할 것

In [ ]:
# [04-01] 필요 패키지 설치
# [목적] opensearch-py와 openai 패키지를 설치합니다
# [주의] 처음 한 번만 실행하면 됩니다. 오류 시 런타임 재시작 후 재실행하세요
# 필요 패키지 설치
!pip install -q opensearch-py openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.7/385.7 kB 7.3 MB/s eta 0:00:00


In [ ]:
# [04-02] 라이브러리 임포트
# [목적] OpenSearch 클라이언트 라이브러리를 불러옵니다
from opensearchpy import OpenSearch

In [ ]:
# [04-03] OpenSearch 접속 정보 설정
# [목적] 서버 주소, 포트, 계정 정보를 변수로 정의합니다
# [주의] STUDENT_NUMBER를 반드시 본인 번호로 변경하세요
# ★ 본인 번호로 변경 (0~30) ★
STUDENT_NUMBER = 0

# 서버 접속 정보
OPENSEARCH_HOST = "sdsos.baeum.ai.kr"
OPENSEARCH_PORT = 443
OPENSEARCH_USER = "sdsrag"
OPENSEARCH_PASS = "Student.Rag1!"

In [ ]:
# [04-04] OpenSearch 클라이언트 생성
# [목적] 접속 정보로 OpenSearch 클라이언트 객체를 만들고 연결을 확인합니다
# [주의] 버전 번호가 출력되지 않으면 접속 정보(호스트/포트/계정)를 재확인하세요
# 클라이언트 생성
opensearch_client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=True,
    verify_certs=True,
    ssl_show_warn=False,
)

# 접속 확인
print(opensearch_client.info()["version"]["number"])

2.19.4


In [ ]:
# [04-05] OpenAI 클라이언트 생성
# [목적] 임베딩 생성에 사용할 OpenAI API 클라이언트를 초기화합니다
# [주의] OPENAI_API_KEY 환경변수가 설정되어 있어야 합니다
from openai import OpenAI

# OpenAI API 키는 각자 본인 키를 사용
# 예: import os; os.environ["OPENAI_API_KEY"] = "sk-..."
openai_client = OpenAI()

In [ ]:
# [04-06] 인덱스명 및 임베딩 설정
# [목적] 검색 대상 인덱스명과 임베딩 모델/차원을 설정합니다
# [개념] text-embedding-3-small은 OpenAI의 경량 임베딩 모델 (1536차원)
# [주의] 다른 학생 번호의 인덱스를 사용하면 안 됩니다
# 본인 번호에 맞는 인덱스 사용 (다른 번호 인덱스 사용 금지)
INDEX_NAME = f"student_{STUDENT_NUMBER:02d}_company_docs"
# 예: student_01_company_docs, student_02_company_docs, ...

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536

In [ ]:
# [04-07] 클러스터 상태 확인
# [목적] OpenSearch 클러스터 상태와 인덱스 목록을 확인합니다
# [주의] status가 green/yellow이면 정상, red이면 관리자에게 문의하세요
# 클러스터 상태 확인
print(opensearch_client.cluster.health()["status"])

# 내 인덱스 목록 확인
print(opensearch_client.cat.indices(index="student_*", format="json"))

print(f"사용 인덱스: {INDEX_NAME}")

yellow
[{'health': 'green', 'status': 'open', 'index': 'student_00_company_docs', 'uuid': 'IfO-8LeqT_aDsW1AzLtIYQ', 'pri': '1', 'rep': '0', 'docs.count': '20', 'docs.deleted': '0', 'store.size': '775.8kb', 'pri.store.size': '775.8kb'}]
사용 인덱스: student_00_company_docs


In [ ]:
# [04-08] get_embedding 함수 정의
# [목적] 텍스트를 벡터(숫자 리스트)로 변환하는 함수를 정의합니다
# [개념] 임베딩: 텍스트의 '의미'를 숫자 벡터로 표현하는 기술
def get_embedding(text: str) -> list[float]:
    """텍스트 임베딩 생성"""
    response = openai_client.embeddings.create(
        input=text,  # 임베딩할 텍스트
        model=EMBEDDING_MODEL,  # text-embedding-3-small 모델
        dimensions=EMBEDDING_DIM,  # 출력 벡터 차원 (1536)
    )
    return response.data[0].embedding  # 첫 번째 결과의 벡터 반환


## 4.2 시맨틱 검색 (Semantic Search)

### 원리

### k-NN (k-Nearest Neighbors)
- 쿼리 벡터와 가장 가까운 k개 문서 벡터 찾기
- 거리 측정: L2 (유클리드), Cosine 등
- HNSW: 근사 최근접 이웃 알고리즘 (빠름)

In [ ]:
# [04-09] semantic_search 함수 정의
# [목적] 쿼리 벡터와 문서 벡터의 유사도로 검색하는 함수를 정의합니다
# [개념] k-NN 검색: 쿼리 벡터에 가장 가까운 k개 문서를 찾는 방식
def semantic_search(query: str, top_k: int = 3) -> list[dict]:
    """시맨틱(벡터) 검색"""
    # 1. 쿼리 텍스트를 벡터로 변환
    query_embedding = get_embedding(query)

    # 2. k-NN 검색 쿼리 구성
    search_body = {
        "query": {
            "knn": {  # k-Nearest Neighbors: 벡터 유사도 기반 검색
                "embedding": {  # 임베딩이 저장된 필드명
                    "vector": query_embedding,  # 쿼리 벡터
                    "k": top_k,  # 가장 가까운 k개 문서 반환
                }
            }
        },
        "size": top_k,  # 최종 반환 문서 수
    }

    # 3. 검색 실행
    response = opensearch_client.search(index=INDEX_NAME, body=search_body)

    # 4. 결과 파싱
    results = []
    for hit in response['hits']['hits']:  # 유사도 높은 순서대로 정렬된 결과
        results.append({
            "id": hit['_id'],  # 문서 고유 ID
            "score": hit['_score'],  # 코사인 유사도 점수 (0~1)
            "title": hit['_source']['title'],
            "category": hit['_source']['category'],
            "content": hit['_source']['content'],
        })

    return results


In [ ]:
# [04-10] 시맨틱 검색 테스트 1: 동의어 처리
# [목적] '진급'이라는 단어가 '승진' 문서를 찾을 수 있는지 확인합니다
# [개념] 시맨틱 검색의 강점 — 동의어/유사어를 의미로 매칭할 수 있음
# 시맨틱 검색 테스트 1: 동의어 처리 (시맨틱 강점!)
print("=== 시맨틱 검색: '진급하고 싶어요' ===")
print("(키워드 검색은 '진급'을 '승진'으로 인식 못함)\n")
results = semantic_search("진급하고 싶어요")  # "진급" ≈ "승진" 의미적으로 유사
for r in results:
    print(f"  [{r['score']:.4f}] {r['title']}")
    print(f"            {r['content'][:60]}...")


=== 시맨틱 검색: '진급하고 싶어요' ===
(키워드 검색은 '진급'을 '승진'으로 인식 못함)

  [0.3980] 리더십 개발 과정
            팀장급 이상을 위한 리더십 개발 과정을 운영합니다. 과정은 코칭 스킬, 성과 관리, 팀 빌딩, 갈등 해결 등...
  [0.3977] 경비 처리 규정
            업무 관련 경비는 법인카드 사용을 원칙으로 합니다. 개인 경비 사용 시에는 영수증과 함께 경비 청구서를 제출...
  [0.3905] 정보보안 정책
            모든 임직원은 정보보안 규정을 준수해야 합니다. 비밀번호는 8자 이상, 특수문자 포함이 필수이며, 90일마다...


In [ ]:
# [04-11] 시맨틱 검색 테스트 2: 자연어 질문
# [목적] 일상 표현('며칠 쉴 수 있나요?')으로 '휴가' 문서를 찾는지 확인합니다
# [개념] 시맨틱 검색은 키워드가 아닌 '의미'로 검색하므로 자연어 질문에 강합니다
# 시맨틱 검색 테스트 2: 자연어 질문 (시맨틱 강점!)
print("=== 시맨틱 검색: '며칠 쉴 수 있나요?' ===")
print("(키워드 검색은 '쉬다'를 '휴가'로 인식 못함)\n")
results = semantic_search("며칠 쉴 수 있나요?")  # "쉬다" ≈ "휴가" 의미적 매칭
for r in results:
    print(f"  [{r['score']:.4f}] {r['title']}")


=== 시맨틱 검색: '며칠 쉴 수 있나요?' ===
(키워드 검색은 '쉬다'를 '휴가'로 인식 못함)

  [0.4062] 휴가 및 근태 관리 규정
  [0.4024] 정보보안 정책
  [0.3976] 리더십 개발 과정


In [ ]:
# [04-12] 시맨틱 검색 테스트 3: 다양한 표현
# [목적] 여러 자연어 표현이 올바른 문서를 찾는지 일괄 테스트합니다
# [개념] 같은 의미의 다양한 표현 → 같은 문서를 찾으면 시맨틱 검색이 잘 동작하는 것
# 시맨틱 검색 테스트 3: 다양한 표현 → 같은 의미
test_queries = [
    ("집에서 일하고 싶어요", "재택근무"),  # 일상 표현 → 공식 용어
    ("월급 언제 나와요?", "급여"),  # 구어체 → 문서 용어
    ("어떻게 올라가요?", "승진"),  # 비유적 표현 → 인사 용어
    ("자격증 따면 뭐가 좋아요?", "자격증 지원"),  # 질문형 → 정책 문서
]

print("=== 다양한 표현으로 검색 (시맨틱 강점) ===\n")
for query, expected in test_queries:  # 각 쿼리와 기대 결과
    results = semantic_search(query, top_k=1)  # 가장 유사한 1건만
    if results:
        print(f"  '{query}'")
        print(f"    → {results[0]['title']} ({results[0]['score']:.4f})")
        print()


=== 다양한 표현으로 검색 (시맨틱 강점) ===

  '집에서 일하고 싶어요'
    → 사내 복지 제도 안내 (0.3929)

  '월급 언제 나와요?'
    → 클라우드 서비스 요금제 안내 (0.4195)

  '어떻게 올라가요?'
    → API 연동 가이드 (0.3775)

  '자격증 따면 뭐가 좋아요?'
    → 정보보안 정책 (0.3902)



## 4.3 키워드 검색 vs 시맨틱 검색 비교

In [ ]:
# [04-13] keyword_search 함수 정의
# [목적] BM25 기반 키워드 검색 함수를 정의합니다
# [개념] multi_match: 여러 필드(title, content)에서 키워드를 매칭하는 쿼리
def keyword_search(query: str, top_k: int = 3) -> list[dict]:
    """키워드(BM25) 검색"""
    search_body = {
        "query": {
            "multi_match": {  # 여러 필드에서 키워드 매칭
                "query": query,
                "fields": ["title^2", "content"],  # title에 2배 가중치
            }
        },
        "size": top_k,  # 반환할 최대 문서 수
    }

    response = opensearch_client.search(index=INDEX_NAME, body=search_body)

    results = []
    for hit in response['hits']['hits']:
        results.append({
            "id": hit['_id'],
            "score": hit['_score'],  # BM25 점수
            "title": hit['_source']['title'],
            "category": hit['_source']['category'],
        })

    return results


In [ ]:
# [04-14] compare_search 함수 정의
# [목적] 같은 검색어로 키워드/시맨틱 결과를 나란히 비교하는 함수를 정의합니다
def compare_search(query: str):
    """키워드 vs 시맨틱 검색 비교"""
    print(f"검색어: '{query}'")
    print("=" * 50)

    # 키워드 검색 (BM25 기반)
    print("\n[BM25 키워드 검색]")
    kw_results = keyword_search(query)
    if kw_results:
        for r in kw_results:
            print(f"  {r['score']:.4f} | {r['title']}")
    else:
        print("  결과 없음")

    # 시맨틱 검색 (벡터 유사도 기반)
    print("\n[시맨틱 검색]")
    sem_results = semantic_search(query)
    for r in sem_results:
        print(f"  {r['score']:.4f} | {r['title']}")

    print()


In [ ]:
# [04-15] 비교 1: 키워드 검색이 강한 케이스
# [목적] 정확한 기술 용어('VPN 설정')는 키워드 검색이 유리한지 확인합니다
# [개념] 키워드 검색은 정확한 단어가 문서에 있을 때 높은 점수를 줍니다
# 비교 1: 키워드 검색이 강한 케이스 (정확한 기술 용어)
compare_search("VPN 설정")  # 정확한 키워드 → BM25가 높은 점수


검색어: 'VPN 설정'

[BM25 키워드 검색]
  5.0179 | 개발 환경 설정 가이드

[시맨틱 검색]
  0.4300 | 정보보안 정책
  0.4253 | API 연동 가이드
  0.4140 | 마이크로서비스 아키텍처 가이드



In [ ]:
# [04-16] 비교 2: 시맨틱 검색이 강한 케이스
# [목적] 동의어/자연어('진급하고 싶어요')는 시맨틱 검색이 유리한지 확인합니다
# 비교 2: 시맨틱 검색이 강한 케이스 (동의어/자연어)
compare_search("진급하고 싶어요")  # "진급" ≈ "승진" → 시맨틱이 의미 매칭


검색어: '진급하고 싶어요'

[BM25 키워드 검색]
  결과 없음

[시맨틱 검색]
  0.3980 | 리더십 개발 과정
  0.3977 | 경비 처리 규정
  0.3905 | 정보보안 정책



In [ ]:
# [04-17] 비교 3: 하이브리드가 필요한 케이스
# [목적] 기술용어 + 의미를 모두 포함한 쿼리에서 두 방식의 한계를 확인합니다
# [개념] 키워드와 시맨틱을 결합한 하이브리드 검색이 필요한 이유를 보여줍니다
# 비교 3: 하이브리드가 필요한 케이스 (기술용어 + 의미)
compare_search("Kubernetes 클러스터 확장 방법")  # 키워드("Kubernetes") + 의미("확장 방법") 모두 필요


검색어: 'Kubernetes 클러스터 확장 방법'

[BM25 키워드 검색]
  2.7322 | 머신러닝 모델 배포 가이드

[시맨틱 검색]
  0.4135 | 마이크로서비스 아키텍처 가이드
  0.4077 | 머신러닝 모델 배포 가이드
  0.4054 | 시스템 장애 대응 매뉴얼



## 4.4 하이브리드 검색 (Hybrid Search)
### 왜 하이브리드인가?
- 키워드 검색: 정확한 용어 매칭에 강함
- 시맨틱 검색: 의미 유사성에 강함
- 둘을 결합하면 상호 보완

### 결합 방식

| 방식 | 설명 |
|------|------|
| RRF | 순위 기반 결합 (랭킹만 사용) |
| 가중 합산 | 점수에 가중치 적용 후 합산 |
| Bool Query | OpenSearch should 절 활용 |

### RRF (Reciprocal Rank Fusion)

```
RRF Score = Σ 1 / (k + rank)

예시 (k=60):
문서 A: 키워드 1위, 시맨틱 3위
  → 1/(60+1) + 1/(60+3) = 0.0164 + 0.0159 = 0.0323

문서 B: 키워드 3위, 시맨틱 1위
  → 1/(60+3) + 1/(60+1) = 0.0159 + 0.0164 = 0.0323

문서 C: 키워드 2위, 시맨틱 2위
  → 1/(60+2) + 1/(60+2) = 0.0161 + 0.0161 = 0.0322
```

→ 두 검색에서 모두 상위인 문서가 높은 점수

In [ ]:
# [04-18] hybrid_search_rrf 함수 정의
# [목적] RRF(Reciprocal Rank Fusion) 방식의 하이브리드 검색 함수를 정의합니다
# [개념] RRF: 각 검색 결과의 '순위'를 기반으로 점수를 합산하는 결합 방식. 점수 스케일이 달라도 안정적
def hybrid_search_rrf(
    query: str,
    top_k: int = 5,
    rrf_k: int = 60  # RRF 상수: 값이 클수록 순위 차이에 덜 민감 (기본값 60)
) -> list[dict]:
    """RRF 기반 하이브리드 검색"""

    # 1. 키워드 검색 (후보를 넉넉히 가져옴)
    kw_results = keyword_search(query, top_k=top_k * 2)

    # 2. 시맨틱 검색 (후보를 넉넉히 가져옴)
    sem_results = semantic_search(query, top_k=top_k * 2)

    # 3. RRF 점수 계산: 각 검색에서의 순위를 기반으로 합산
    rrf_scores = {}  # 문서별 RRF 합산 점수
    doc_data = {}  # 문서별 메타데이터 저장

    # 키워드 검색 결과의 RRF 점수 누적
    for rank, doc in enumerate(kw_results, start=1):  # rank는 1부터 시작
        doc_id = doc['id']
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (rrf_k + rank)  # RRF 공식: 1/(k+rank)
        doc_data[doc_id] = doc

    # 시맨틱 검색 결과의 RRF 점수 누적
    for rank, doc in enumerate(sem_results, start=1):
        doc_id = doc['id']
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (rrf_k + rank)  # 기존 점수에 합산
        doc_data[doc_id] = doc

    # 4. RRF 점수 내림차순 정렬 (양쪽에서 모두 상위일수록 높은 점수)
    sorted_ids = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)

    # 5. 상위 top_k개 결과 반환
    results = []
    for doc_id in sorted_ids[:top_k]:
        doc = doc_data[doc_id]
        doc['rrf_score'] = rrf_scores[doc_id]  # 최종 RRF 점수 추가
        results.append(doc)

    return results


In [ ]:
# [04-19] RRF 하이브리드 검색 테스트
# [목적] 키워드 강점/시맨틱 강점/하이브리드 케이스를 RRF로 검색해 결과를 확인합니다
# [주의] rrf_score가 양쪽 검색에서 모두 상위인 문서일수록 높게 나옵니다
# RRF 하이브리드 검색 테스트
print("=== RRF 하이브리드 검색 ===\n")

test_queries = [
    "VPN 접속 방법",           # 키워드 강점 케이스
    "집에서 일하고 싶어요",     # 시맨틱 강점 케이스
    "Kubernetes 오토스케일링",  # 양쪽 모두 필요한 케이스
]

for query in test_queries:
    print(f"검색어: '{query}'")
    results = hybrid_search_rrf(query, top_k=2)
    for r in results:
        print(f"  [RRF: {r['rrf_score']:.4f}] {r['title']}")  # RRF 합산 점수 출력
    print()


=== RRF 하이브리드 검색 ===

검색어: 'VPN 접속 방법'
  [RRF: 0.0164] 정보보안 정책
  [RRF: 0.0161] API 연동 가이드

검색어: '집에서 일하고 싶어요'
  [RRF: 0.0164] 사내 복지 제도 안내
  [RRF: 0.0161] 정보보안 정책

검색어: 'Kubernetes 오토스케일링'
  [RRF: 0.0328] 머신러닝 모델 배포 가이드
  [RRF: 0.0161] 마이크로서비스 아키텍처 가이드



### Bool Query 기반 하이브리드

OpenSearch의 should 절로 두 쿼리 결합

In [ ]:
# [04-20] hybrid_search_bool 함수 정의
# [목적] OpenSearch Bool Query의 should절로 키워드+시맨틱을 결합하는 함수를 정의합니다
# [개념] Bool should: 두 쿼리 중 하나만 만족해도 결과에 포함되며, 둘 다 만족하면 점수가 높아짐
def hybrid_search_bool(
    query: str,
    top_k: int = 5,
    keyword_weight: float = 0.3,  # 키워드 검색 가중치 (기본 30%)
    semantic_weight: float = 0.7,  # 시맨틱 검색 가중치 (기본 70%)
) -> list[dict]:
    """Bool Query 기반 하이브리드 검색"""

    query_embedding = get_embedding(query)  # 쿼리를 벡터로 변환

    # Bool should절: 두 쿼리 중 하나만 매칭되어도 결과 포함, 둘 다면 점수 합산
    search_body = {
        "query": {
            "bool": {
                "should": [  # should: OR 조건 + 점수 합산
                    # 키워드 검색 파트
                    {
                        "multi_match": {
                            "query": query,
                            "fields": ["title^2", "content"],
                            "boost": keyword_weight,  # BM25 점수에 가중치 적용
                        }
                    },
                    # 시맨틱 검색 파트
                    {
                        "knn": {  # k-NN 벡터 검색
                            "embedding": {
                                "vector": query_embedding,  # 쿼리 벡터
                                "k": top_k,  # 최근접 k개
                            }
                        }
                    },
                ]
            }
        },
        "size": top_k,
    }

    response = opensearch_client.search(index=INDEX_NAME, body=search_body)

    results = []
    for hit in response['hits']['hits']:
        results.append({
            "id": hit['_id'],
            "score": hit['_score'],  # 키워드+시맨틱 합산 점수
            "title": hit['_source']['title'],
            "category": hit['_source']['category'],
            "content": hit['_source']['content'],
        })

    return results


In [ ]:
# [04-21] Bool 하이브리드 검색 테스트
# [목적] Bool 방식 하이브리드 검색의 결과를 확인합니다
# Bool 하이브리드 검색 테스트
print("=== Bool 하이브리드 검색: '휴가 신청 방법' ===")
results = hybrid_search_bool("휴가 신청 방법")
for r in results:
    print(f"  [{r['score']:.4f}] {r['title']}")

=== Bool 하이브리드 검색: '휴가 신청 방법' ===
  [1.8388] 휴가 및 근태 관리 규정
  [0.4417] 경비 처리 규정
  [0.4145] 사내 복지 제도 안내
  [0.4110] 정보보안 정책
  [0.4040] 고객사 관리 정책


## 4.5 검색 파이프라인 통합

In [ ]:
# [04-22] SearchPipeline 클래스 정의
# [목적] keyword/semantic/hybrid 검색을 하나의 인터페이스로 통합하는 클래스를 정의합니다
# [개념] method 파라미터로 검색 방식을 전환할 수 있어 비교 실험에 편리합니다
class SearchPipeline:
    """검색 파이프라인 클래스"""

    def __init__(self, index_name: str):
        self.index_name = index_name  # 검색 대상 인덱스명

    def search(
        self,
        query: str,
        method: str = "hybrid",  # 검색 방식: keyword / semantic / hybrid
        top_k: int = 3,
        category: str = None,  # 선택적 카테고리 필터
    ) -> list[dict]:
        """통합 검색 메서드"""

        # method에 따라 적절한 검색 함수 호출
        if method == "keyword":
            results = self._keyword_search(query, top_k)
        elif method == "semantic":
            results = self._semantic_search(query, top_k)
        elif method == "hybrid":
            results = self._hybrid_search(query, top_k)
        else:
            raise ValueError(f"Unknown method: {method}")

        # 카테고리 필터링 (후처리 방식: 검색 후 결과에서 필터)
        if category:
            results = [r for r in results if r['category'] == category]

        return results

    def _keyword_search(self, query: str, top_k: int) -> list[dict]:
        return keyword_search(query, top_k)  # 기존 BM25 검색 함수 위임

    def _semantic_search(self, query: str, top_k: int) -> list[dict]:
        return semantic_search(query, top_k)  # 기존 벡터 검색 함수 위임

    def _hybrid_search(self, query: str, top_k: int) -> list[dict]:
        return hybrid_search_bool(query, top_k)  # Bool 기반 하이브리드 검색 위임


In [ ]:
# [04-23] SearchPipeline 테스트
# [목적] 파이프라인으로 3가지 검색 방식의 결과를 한눈에 비교합니다
# [주의] 각 케이스에서 어떤 방식이 더 좋은 결과를 내는지 비교해 보세요
# 파이프라인 사용 - 검색 방식별 비교
pipeline = SearchPipeline(INDEX_NAME)

# 각 케이스에서 어떤 방식이 더 좋은 결과를 내는지 비교
test_cases = [
    ("SonarQube 사용법", "키워드 강점"),  # 정확한 도구명 → BM25 유리
    ("며칠 쉴 수 있나요?", "시맨틱 강점"),  # 자연어 → 벡터 유리
    ("클라우드 보안 설정", "하이브리드"),  # 기술용어+의미 → 결합 유리
]

for query, case_type in test_cases:
    print(f"=== {case_type}: '{query}' ===")
    for method in ["keyword", "semantic", "hybrid"]:  # 3가지 방식 순차 비교
        results = pipeline.search(query, method=method, top_k=1)  # 최상위 1건만
        if results:
            print(f"  {method:8s} → {results[0]['title']}")
        else:
            print(f"  {method:8s} → 결과 없음")
    print()


=== 키워드 강점: 'SonarQube 사용법' ===
  keyword  → 결과 없음
  semantic → 개발 환경 설정 가이드
  hybrid   → 개발 환경 설정 가이드

=== 시맨틱 강점: '며칠 쉴 수 있나요?' ===
  keyword  → API 연동 가이드
  semantic → 휴가 및 근태 관리 규정
  hybrid   → API 연동 가이드

=== 하이브리드: '클라우드 보안 설정' ===
  keyword  → 클라우드 서비스 요금제 안내
  semantic → 클라우드 서비스 요금제 안내
  hybrid   → 클라우드 서비스 요금제 안내



## 4.6 검색 품질 평가 (간단)
### 평가 지표
- **Precision@K**: 상위 K개 중 관련 문서 비율
- **Recall@K**: 전체 관련 문서 중 상위 K개에 포함된 비율
- **MRR (Mean Reciprocal Rank)**: 첫 관련 문서 순위의 역수 평균

## 4.7 정리
### 학습 내용

| 검색 방식 | 장점 | 단점 |
|----------|------|------|
| 키워드 | 정확한 용어 매칭 | 동의어 X, 의미 X |
| 시맨틱 | 의미 유사성, 동의어 | 정확 매칭 약함 |
| 하이브리드 | 두 장점 결합 | 복잡도 증가 |

### 하이브리드 검색 방식
- **RRF**: 순위 기반 결합, 점수 스케일 무관
- **Bool Query**: OpenSearch 네이티브, 가중치 조절 가능

### 다음 학습
- RAG 파이프라인 구현
- 검색 결과 + LLM 연동